In [1]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px

In [48]:
train_path = Path("rogii-wellbore-geology-prediction/train")
train_files = list(train_path.glob("*"))
train_filenames = [f.name.strip(".png") for f in train_path.glob("*.png")]

test_path = Path("rogii-wellbore-geology-prediction/test")
test_files = list(test_path.glob("*"))
test_filenames = [f.name.strip("__horizontal_well.csv") for f in test_path.glob("*__horizontal_well.csv")]

In [50]:
train_data = {}
for filename in tqdm(train_filenames):
    train_data[filename] = {}
    train_data[filename]["Well"] = pd.read_csv(train_path/(filename+"__horizontal_well.csv"))
    train_data[filename]["TypeWell"] = pd.read_csv(train_path/(filename+"__typewell.csv"))

    


  0%|          | 0/773 [00:00<?, ?it/s]

In [51]:
test_data = {}
for filename in tqdm(test_filenames):
    test_data[filename] = {}
    test_data[filename]["Well"] = pd.read_csv(test_path/(filename+"__horizontal_well.csv"))
    test_data[filename]["TypeWell"] = pd.read_csv(test_path/(filename+"__typewell.csv"))

    


  0%|          | 0/3 [00:00<?, ?it/s]

In [153]:
# filename="0e5e560d"
filename = list(train_data.keys())[0]
well_data = train_data[filename]["Well"]
pred_start = well_data[well_data["TVT_input"].isna()].index[0]
typewell_data = train_data[filename]["TypeWell"].set_index("TVT")

In [155]:
filename

'000d7d20'

In [156]:
import plotly.graph_objects as go

fig = px.scatter_3d(well_data, x="X", y="Y", z="Z", color="TVT").update_layout(height=800, width=800)
fig.update_traces(marker=dict(size=2))

start_point = well_data.loc[pred_start]
fig.add_trace(go.Scatter3d(
    x=[start_point["X"]],
    y=[start_point["Y"]],
    z=[start_point["Z"]],
    mode="markers",
    marker=dict(color="red", size=8),
    name="Prediction Start",
    showlegend=False
))

fig

# No Pred

In [157]:
ytrue = well_data.loc[pred_start:,"TVT"]
ypred = well_data.loc[pred_start-1,"TVT_input"]*np.ones(ytrue.shape)

loss = ((ytrue-ypred)**2).mean()**0.5
print(loss)

7.454442102844319


# Z Pred

In [169]:
px.line(well_data.loc[pred_start:,], x="MD",y=["TVT"])

In [159]:
px.line(well_data.loc[pred_start:,], x="MD",y=["Z"])